<a href="https://colab.research.google.com/github/bsudharsh1979/NATcolab/blob/main/notebooks/ALM_assessment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**NATcolab** · NVIDIA NeMo Agent Toolkit · *Asset Lifecycle Management with Agentic AI* — Google Colab edition (hosted NVIDIA NIMs)

# Assessment: Asset Lifecycle Management
In this assessment you will:

1. **Build a new NAT tool** — Add a moving average overlay to the line chart tool
2. **Test it end-to-end** — Run a query using your new tool through the Agent UI
3. **Capture and analyze the trace** — Export Phoenix traces and analyze the agent's behavior

> **How to use this (completed) notebook:** run every code cell top to bottom, waiting for each to finish.
> Step 1 writes the completed tool, wires it into the agent, restarts the backend and runs the test query for you;
> Steps 2–6 produce and validate the four graded deliverables in `Assessment/output/`.


> **🟢 Colab edition.** This is the Google Colab adaptation of the NVIDIA DLI lab notebook. Instead of the lab's Docker containers it uses:
> - **Hosted NVIDIA NIMs** on build.nvidia.com — `nvidia/nemotron-3-ultra-550b-a55b` (planning, tool calling, text-to-SQL), `nvidia/nemotron-3.5-lightning-30b-a3b` (code generation), `nvidia/nemotron-3-nano-omni-30b-a3b-reasoning` (multimodal judge), `nvidia/nemotron-3.5-embed-1b` (embeddings)
> - **MOMENT-1-small**, an open-source time-series foundation model that runs locally, in place of the self-hosted NV-Tesseract NIM
> - a local Python **code sandbox** and a **Phoenix** server started from the notebook
>
> **Before you run:** add your NVIDIA API key (get one at [build.nvidia.com](https://build.nvidia.com)) as a Colab secret named `NVIDIA_API_KEY` — 🔑 icon in the left sidebar → *Add new secret* → enable *Notebook access*. A CPU runtime is enough.
> Colab gives every notebook its own VM, so run the **setup cell below at the top of every notebook** (≈5–8 min on a fresh runtime, seconds afterwards). Paths that were `/dli/task/...` in the lab are `/content/NATcolab/...` here.

> **Prerequisites:** none in Colab — the setup cell installs the agent package and builds the database (Notebooks 0/1), and Step 0 starts Phoenix (Notebook 3) in this runtime.

### Table of Contents
[The Problem](#The-Problem)<br>
[Scoring](#Scoring)<br>
[Step 0: Pre-Flight Check](#Step-0:-Pre-Flight-Check)<br>
[Step 1: Complete, Wire, and Test Your Tool](#Step-1:-Complete,-Wire,-and-Test-Your-Tool)<br>
[Step 2: Save Your Code and JSON Output](#Step-2:-Save-Your-Code-and-JSON-Output)<br>
[Step 3: Export the Phoenix Trace](#Step-3:-Export-the-Phoenix-Trace)<br>
[Step 4: Verify Trace Export](#Step-4:-Verify-Trace-Export)<br>
[Step 5: Analyze the Trace](#Step-5:-Analyze-the-Trace)<br>
[Step 6: Final Verification](#Step-6:-Final-Verification)<br>
[Step 7: Submit Your Assessment](#Step-7:-Submit-Your-Assessment)

---
# The Problem

In this assessment, your task is to build a new NAT tool, test it end-to-end, and analyze the agent's behavior using Phoenix tracing:

Your tool is called **`plot_line_chart_with_ma_tool`** with the following features:
1. Loads sensor data from a JSON file (same as the existing line chart tool)
2. Plots the raw data as a line chart
3. **Computes a moving average** of the y-axis column
4. **Overlays the moving average** as a second line (red) on the same chart
5. **Saves a JSON file** with the raw data and a column named **`moving_average`**


### Starter File

[Assessment/plot_line_chart_with_ma_tool.py](plot_line_chart_with_ma_tool.py)<br>This is based on the existing `plot_line_chart_tool.py` with **4 `ADD YOUR CODE HERE` blocks**:

| Block | What to Add | Hint |
|-------|-------------|------|
| 1 of 4 | `window_size` parameter in input schema | `window_size: int = Field(description="...", default=10)` |
| 2 of 4 | Compute moving average using pandas | `df_sorted['moving_average'] = df_sorted[col].rolling(window=N).mean()` |
| 3 of 4 | Add second Plotly trace for moving average | Follow the pattern of the raw data trace above it |
| 4 of 4 | Save JSON with raw + moving average data | `output_df.to_json(path, orient='records', indent=2)` |

### Reference Files

Study these files for patterns:
- [asset_lifecycle_management_agent/src/asset_lifecycle_management_agent/plotting/plot_line_chart_tool.py](../asset_lifecycle_management_agent/src/asset_lifecycle_management_agent/plotting/plot_line_chart_tool.py)
- [asset_lifecycle_management_agent/src/asset_lifecycle_management_agent/plotting/plot_utils.py](../asset_lifecycle_management_agent/src/asset_lifecycle_management_agent/plotting/plot_utils.py)

**Important naming requirements:**
- The tool must be named exactly: **`plot_line_chart_with_ma_tool`**
- The moving average column in the output JSON must be named exactly: **`moving_average`**


---
# Scoring
You will be assessed on your ability to successfully implement the tool with the following scoring.

### Rubric

All deliverables will be placed in the `Assessment/output/` folder for grading:

| Deliverable                                 | Graded                                                    | Points |
|--------------------------------------|-----------------------------------------------------------|--------|
| `plot_line_chart_with_ma_tool.py` | Your completed tool source code                          |50   |
| `moving_average_output.json` | JSON output with raw data + `moving_average` column|  30|
| `traces.json` | Raw Phoenix trace export (JSON) | 10|
| `trace_analysis.json` | Your analysis of the trace|  10|
| **Total** | | **100** |

Once you are confident that you've built a correct tool and saved the deliverables, follow the instructions for submission at the end of the notebook.

**Have Fun!**

---
# Step 0: Pre-Flight Check

Before starting, reset the config and verify the models and Phoenix are up and running.

In [ ]:
# ⚙️ NATcolab setup — run first (≈5–8 min on a fresh runtime; a few seconds if already done in this session)
# Clones github.com/bsudharsh1979/NATcolab, creates the NAT virtual env (nvidia-nat 1.2.1 + ALM agent),
# builds the NASA turbofan database, trains the RUL model and checks the hosted NVIDIA NIMs.
import os, sys, subprocess
ALM_HOME = os.environ.get("ALM_HOME", "/content/NATcolab")
if not os.path.exists(f"{ALM_HOME}/colab_bootstrap.py"):
    subprocess.run(["git", "clone", "-q", "https://github.com/bsudharsh1979/NATcolab.git", ALM_HOME], check=True)
sys.path.insert(0, ALM_HOME)
os.chdir(ALM_HOME)

import colab_bootstrap
colab_bootstrap.bootstrap()

In [ ]:
from nat_client import NATClient, link
client = NATClient()

> **Reset codebase:** If you need to restore all source files and config to their original state (e.g. after testing a solution), run the cell below.

In [ ]:
client.reset()

### Check Backend Models (hosted NVIDIA NIMs)

In [ ]:
import colab_bootstrap
assert colab_bootstrap.check_nims(), "Some NIM endpoints did not answer — check your NVIDIA_API_KEY / quota"

### Start Phoenix Observability

In the lab, Phoenix is already running from the **Notebook 3** prerequisite. Every Colab notebook runs on its own VM, so start it here (idempotent):

In [ ]:
client.start_phoenix()

### Agent Front-End

The NAT Toolkit UI is not used in Colab — run queries with `client.query(...)`; pass `show_steps=True` to see the agent's plan and tool calls inline.

### Open Phoenix Dashboard
You can skip this if the Phoenix UI is already open.

In [ ]:
link(6006, "Open Phoenix Dashboard")

---
# Step 1: Complete, Wire, and Test Your Tool

### 1a. Complete the Starter File

The cell below writes the **completed** `plot_line_chart_with_ma_tool.py` (all 4 `ADD YOUR CODE HERE` blocks filled in) to
`/content/NATcolab/Assessment/plot_line_chart_with_ma_tool.py`, overwriting the starter file. A backup of the starter is kept as
`plot_line_chart_with_ma_tool.py.starter.bak` (first run only).

| Block | Implemented as |
|-------|----------------|
| 1 of 4 | `window_size: int = Field(description=..., default=10)` in the input schema |
| 2 of 4 | `df_sorted['moving_average'] = df_sorted[y_axis_column].rolling(window=window_size).mean()` |
| 3 of 4 | Second `go.Scatter` trace, `line=dict(color='red', width=3)`, named `Moving Average (window=N)` |
| 4 of 4 | `output_df[[x, y, 'moving_average']].to_json(<output_folder>/moving_average_output.json, orient='records', indent=2)` |

In [ ]:
import os, shutil

starter_path = f"{ALM_HOME}/Assessment/plot_line_chart_with_ma_tool.py"
backup_path = starter_path + ".starter.bak"
os.makedirs(os.path.dirname(starter_path), exist_ok=True)
if os.path.exists(starter_path) and not os.path.exists(backup_path):
    shutil.copy(starter_path, backup_path)
    print(f"Backed up starter -> {backup_path}")

COMPLETED_TOOL_SOURCE = r'''# SPDX-FileCopyrightText: Copyright (c) 2023-2024 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

"""
plot_line_chart_with_ma_tool
============================

NeMo Agent Toolkit (NAT) tool that:

1. Loads sensor data from a JSON file (same as ``plot_line_chart_tool``)
2. Plots the raw data as a line chart (blue)
3. Computes a moving average of the y-axis column
4. Overlays the moving average as a second line (red) on the same chart
5. Saves a JSON file (``moving_average_output.json``) with the raw data and a
   column named ``moving_average``
"""

import json
import logging
import os

from pydantic import BaseModel, Field

from nat.builder.builder import Builder
from nat.builder.function_info import FunctionInfo
from nat.cli.register_workflow import register_function
from nat.data_models.function import FunctionBaseConfig

logger = logging.getLogger(__name__)

# Name of the JSON file written by this tool (graded deliverable)
MOVING_AVERAGE_OUTPUT_FILENAME = "moving_average_output.json"


def verify_json_path(file_path: str, output_folder: str = None) -> str:
    """
    Verify that the input is a valid path to a JSON file.

    Args:
        file_path (str): Path to verify
        output_folder (str): Output folder to use as base for relative paths

    Returns:
        str: Verified (resolved) file path

    Raises:
        ValueError: If input is not a string or not a JSON file
        FileNotFoundError: If file does not exist
    """
    if not isinstance(file_path, str):
        raise ValueError("Input must be a string path to a JSON file")

    if not file_path.lower().endswith('.json'):
        raise ValueError("Input must be a path to a JSON file (ending with .json)")

    # Resolve path relative to output folder if provided
    if output_folder:
        from .plot_utils import resolve_relative_path
        resolved_path = resolve_relative_path(file_path, output_folder)
    else:
        resolved_path = file_path

    if not os.path.exists(resolved_path):
        raise FileNotFoundError(f"JSON file not found at path: {file_path}")

    try:
        with open(resolved_path, 'r') as f:
            json.load(f)  # Verify file contains valid JSON
    except json.JSONDecodeError:
        raise ValueError(f"File at {file_path} does not contain valid JSON data")

    return resolved_path


class PlotLineChartWithMAToolConfig(FunctionBaseConfig, name="plot_line_chart_with_ma_tool"):
    """
    NeMo Agent Toolkit function to plot a line chart with a moving average overlay.
    """
    output_folder: str = Field(description="The path to the output folder to save plots.",
                               default="./output_data")


@register_function(config_type=PlotLineChartWithMAToolConfig)
async def plot_line_chart_with_ma_tool(
    config: PlotLineChartWithMAToolConfig, builder: Builder
):
    class PlotLineChartWithMAInputSchema(BaseModel):
        data_json_path: str = Field(description="The path to the JSON file containing the data")
        x_axis_column: str = Field(description="The column name for x-axis data", default="time_in_cycles")
        y_axis_column: str = Field(description="The column name for y-axis data", default="RUL")
        plot_title: str = Field(description="The title for the plot", default="Line Chart with Moving Average")
        # ===================== ADD YOUR CODE HERE (1 of 4) =====================
        # Add a window_size parameter so the agent can control the moving-average window.
        window_size: int = Field(description="The window size (number of data points) used to compute the moving average",
                                 default=10)
        # =======================================================================

    from .plot_utils import load_data_from_json, save_plotly_as_png

    async def _response_fn(data_json_path: str,
                           x_axis_column: str,
                           y_axis_column: str,
                           plot_title: str,
                           window_size: int = 10) -> str:
        """
        Process the input message and generate a line chart with a moving average overlay.
        """
        import plotly.graph_objects as go
        import plotly.offline as pyo

        try:
            data_json_path = verify_json_path(data_json_path, config.output_folder)

            # Load data to validate columns exist
            df = load_data_from_json(data_json_path, config.output_folder)
            if df is None or df.empty:
                return "Could not load data or data is empty from the provided JSON file"

            # Check required columns
            required_columns = [x_axis_column, y_axis_column]
            missing_columns = [col for col in required_columns if col not in df.columns]
            if missing_columns:
                return (f"Data from {data_json_path} must contain columns: {required_columns}. "
                        f"Missing: {missing_columns}")

            # Sanitize the window size (must be a positive integer no larger than the data length)
            try:
                window_size = int(window_size)
            except (TypeError, ValueError):
                window_size = 10
            window_size = max(1, min(window_size, len(df)))

            # Sort by x-axis column for proper line plotting
            df_sorted = df.sort_values(x_axis_column).reset_index(drop=True)

            # ===================== ADD YOUR CODE HERE (2 of 4) =====================
            # Compute the moving average of the y-axis column using pandas.
            df_sorted['moving_average'] = df_sorted[y_axis_column].rolling(window=window_size).mean()
            # =======================================================================

            # Create the figure
            fig = go.Figure()

            # Raw data trace (blue)
            fig.add_trace(go.Scatter(
                x=df_sorted[x_axis_column],
                y=df_sorted[y_axis_column],
                mode='lines+markers',
                name=y_axis_column,
                line=dict(color='#1f77b4', width=3),
                marker=dict(size=6, color='#1f77b4'),
                hovertemplate=f'<b>{x_axis_column}</b>: %{{x}}<br>' +
                              f'<b>{y_axis_column}</b>: %{{y:.2f}}<br>' +
                              '<extra></extra>'
            ))

            # ===================== ADD YOUR CODE HERE (3 of 4) =====================
            # Moving average trace (red) overlaid on the same chart.
            fig.add_trace(go.Scatter(
                x=df_sorted[x_axis_column],
                y=df_sorted['moving_average'],
                mode='lines',
                name=f'Moving Average (window={window_size})',
                line=dict(color='red', width=3),
                hovertemplate=f'<b>{x_axis_column}</b>: %{{x}}<br>' +
                              '<b>Moving Average</b>: %{y:.2f}<br>' +
                              '<extra></extra>'
            ))
            # =======================================================================

            # Update layout
            fig.update_layout(
                title=dict(text=plot_title, x=0.5, font=dict(size=16)),
                xaxis=dict(title=dict(text=x_axis_column, font=dict(size=14)),
                           gridcolor='lightgray', gridwidth=0.5),
                yaxis=dict(title=dict(text=y_axis_column, font=dict(size=14)),
                           gridcolor='lightgray', gridwidth=0.5),
                width=650, height=450, plot_bgcolor='white',
                showlegend=True,
                legend=dict(x=1, y=0, xanchor='right', yanchor='bottom',
                            bgcolor='rgba(255,255,255,0.8)', bordercolor='gray', borderwidth=1),
                hovermode='closest'
            )

            # Set y-axis range
            y_min = df_sorted[y_axis_column].min()
            y_max = df_sorted[y_axis_column].max()
            y_range = y_max - y_min
            if y_range > 0:
                fig.update_yaxes(range=[y_min - y_range * 0.05, y_max + y_range * 0.05])

            # Save files
            os.makedirs(config.output_folder, exist_ok=True)

            # HTML file
            html_filepath = os.path.join(
                config.output_folder, f"line_chart_with_ma_{x_axis_column}_vs_{y_axis_column}.html")
            html_content = pyo.plot(fig, output_type='div', include_plotlyjs=True)
            full_html = f"""
<!DOCTYPE html>
<html>
<head>
    <title>{plot_title}</title>
    <meta charset="utf-8">
</head>
<body>
    {html_content}
</body>
</html>
"""
            with open(html_filepath, 'w', encoding='utf-8') as f:
                f.write(full_html)
            logger.info(f"Line chart with moving average HTML saved: {html_filepath}")

            # PNG file
            png_filepath = os.path.join(
                config.output_folder, f"line_chart_with_ma_{x_axis_column}_vs_{y_axis_column}.png")
            try:
                png_success = save_plotly_as_png(fig, png_filepath, width=650, height=450)
            except Exception as png_err:  # pragma: no cover - PNG is best-effort
                logger.error(f"PNG generation failed: {png_err}")
                png_success = False
            if not png_success:
                png_filepath = None

            # ===================== ADD YOUR CODE HERE (4 of 4) =====================
            # Save the raw data plus the moving average column to a JSON file.
            output_df = df_sorted[[x_axis_column, y_axis_column, 'moving_average']]
            json_output_path = os.path.join(config.output_folder, MOVING_AVERAGE_OUTPUT_FILENAME)
            output_df.to_json(json_output_path, orient='records', indent=2)
            logger.info(f"Moving average JSON saved: {json_output_path}")
            # =======================================================================

            # Build file information for response (relative paths from output folder)
            html_relpath = os.path.relpath(html_filepath, config.output_folder)
            json_relpath = os.path.relpath(json_output_path, config.output_folder)
            file_info = f"- HTML File: {html_relpath}"
            if png_filepath:
                png_relpath = os.path.relpath(png_filepath, config.output_folder)
                file_info += f"\n- PNG File: {png_relpath}"
            file_info += f"\n- JSON File (raw data + moving_average column): {json_relpath}"

            # Return a clear completion message that the LLM will understand
            return f"""TASK COMPLETED SUCCESSFULLY

Line chart with moving average overlay has been generated and saved in multiple formats.

Chart Details:
- Type: Line chart with markers (Plotly) + moving average overlay (red line)
- X-axis: {x_axis_column}
- Y-axis: {y_axis_column}
- Moving average window size: {window_size}
- Title: {plot_title}
{file_info}

✅ CHART GENERATION COMPLETE - NO FURTHER ACTION NEEDED"""

        except FileNotFoundError as e:
            error_msg = f"Required data file ('{data_json_path}') not found for line chart: {e}"
            logger.error(error_msg)
            return error_msg
        except KeyError as ke:
            error_msg = f"Missing required columns in '{data_json_path}' for line chart: {ke}"
            logger.error(error_msg)
            return error_msg
        except ValueError as ve:
            error_msg = f"Data validation error for line chart: {ve}"
            logger.error(error_msg)
            return error_msg
        except Exception as e:
            error_msg = f"Error generating line chart with moving average: {e}"
            logger.error(error_msg)
            return error_msg

    prompt = """
    Generate an interactive line chart from JSON data using Plotly, with a moving average overlay.

    Input:
    - data_json_path: Path to the JSON file containing the data
    - x_axis_column: Column name for x-axis data
    - y_axis_column: Column name for y-axis data
    - plot_title: Title for the plot
    - window_size: Window size (number of points) for the moving average (default 10)

    Output:
    - HTML file containing the interactive line chart (raw data in blue, moving average in red)
    - PNG file containing the static line chart
    - JSON file (moving_average_output.json) containing the raw data plus a 'moving_average' column
    """
    yield FunctionInfo.from_fn(_response_fn,
                               input_schema=PlotLineChartWithMAInputSchema,
                               description=prompt)
    try:
        pass
    except GeneratorExit:
        logger.info("Plot line chart with moving average function exited early!")
    finally:
        logger.info("Cleaning up plot_line_chart_with_ma_tool workflow.")
'''

with open(starter_path, "w", encoding="utf-8") as f:
    f.write(COMPLETED_TOOL_SOURCE)

# Sanity-check the written file
import ast
ast.parse(COMPLETED_TOOL_SOURCE)
checks = {
    'tool name registered':      'name="plot_line_chart_with_ma_tool"' in COMPLETED_TOOL_SOURCE,
    'window_size in schema':     'window_size: int = Field(' in COMPLETED_TOOL_SOURCE,
    'moving_average via rolling': ".rolling(window=window_size).mean()" in COMPLETED_TOOL_SOURCE,
    'red overlay trace':         "color='red'" in COMPLETED_TOOL_SOURCE,
    'JSON output file':          'moving_average_output.json' in COMPLETED_TOOL_SOURCE and ".to_json(" in COMPLETED_TOOL_SOURCE,
}
for k, v in checks.items():
    print(('✅' if v else '❌'), k)
assert all(checks.values()), 'Completed tool file failed a sanity check'
print(f"\n✅ Wrote completed tool ({len(COMPLETED_TOOL_SOURCE)} chars) -> {starter_path}")

### 1b. Wire the Tool Into the Agent

**Copy your file into the agent source tree:**

In [ ]:
import shutil
shutil.copy(f"{ALM_HOME}/Assessment/plot_line_chart_with_ma_tool.py",
            f"{ALM_HOME}/asset_lifecycle_management_agent/src/asset_lifecycle_management_agent/plotting/")
print("✅ copied plot_line_chart_with_ma_tool.py into plotting/")

**Register the tool and update the config.** The cell below performs all of the required edits automatically and is idempotent (safe to re-run):

1. `plotting/__init__.py` — adds `from . import plot_line_chart_with_ma_tool` and appends `"plot_line_chart_with_ma_tool"` to `__all__`
2. `register.py` — adds `from .plotting import plot_line_chart_with_ma_tool`
3. `configs/config-reasoning.yml`:
   - adds the `plot_line_chart_with_ma` function (`_type: plot_line_chart_with_ma_tool`, `output_folder: "agent_workspace"`)
   - adds `plot_line_chart_with_ma` to `data_analysis_assistant.tool_names`
   - adds the moving-average instruction under **Analysis and Plotting Tools** in the `system_prompt`
   - enables Phoenix tracing (`general.telemetry.tracing.phoenix` → `http://localhost:6006/v1/traces`, project `asset-lifecycle-management`)

Every edit prints ✅ (applied), ℹ️ (already present) or ❌ (pattern not found → apply that one edit by hand using the instructions in the expandable section below).

In [ ]:
import re, os, shutil

SRC_DIR = f"{ALM_HOME}/asset_lifecycle_management_agent/src/asset_lifecycle_management_agent"
INIT_PY = f"{SRC_DIR}/plotting/__init__.py"
REGISTER_PY = f"{SRC_DIR}/register.py"
CONFIG_YML = f"{ALM_HOME}/configs/config-reasoning.yml"

TOOL_MODULE = "plot_line_chart_with_ma_tool"
TOOL_NAME = "plot_line_chart_with_ma"
PROMPT_LINE = ("- plot_line_chart_with_ma: to plot a line chart WITH a moving average overlay. "
               "Use this tool whenever the user asks for a moving average plot. "
               "Do NOT use code_generation_assistant for moving average tasks.")

def read(p):  return open(p, encoding="utf-8").read()
def write(p, s):
    bak = p + ".pre_assessment.bak"
    if not os.path.exists(bak):
        shutil.copy(p, bak)
    open(p, "w", encoding="utf-8").write(s)

def report(label, status):
    print({"applied": "✅", "present": "ℹ️ ", "missing": "❌"}[status], label, "" if status != "missing" else "→ apply by hand")
    return status != "missing"

ok = True

# ---------------------------------------------------------------- 1. plotting/__init__.py
s = read(INIT_PY)
if f"from . import {TOOL_MODULE}" in s:
    ok &= report("__init__.py import", "present")
else:
    imports = list(re.finditer(r"^from \. import \w+[ \t]*$", s, flags=re.M))
    if imports:
        last = imports[-1]
        s = s[:last.end()] + f"\nfrom . import {TOOL_MODULE}" + s[last.end():]
    else:
        s = s.rstrip("\n") + f"\nfrom . import {TOOL_MODULE}\n"
    ok &= report("__init__.py import", "applied")
if re.search(rf'"{TOOL_MODULE}"', s):
    ok &= report("__init__.py __all__ entry", "present")
else:
    m = re.search(r"__all__\s*=\s*\[(.*?)\]", s, flags=re.S)
    if m:
        body = m.group(1).rstrip()
        if body and not body.endswith(","):
            body += ","
        new_body = body + f'\n    "{TOOL_MODULE}",\n'
        s = s[:m.start(1)] + new_body + s[m.end(1):]
        ok &= report("__init__.py __all__ entry", "applied")
    else:
        s = s.rstrip("\n") + f'\n\n__all__ = ["{TOOL_MODULE}"]\n'
        ok &= report("__init__.py __all__ entry (created __all__)", "applied")
write(INIT_PY, s)

# ---------------------------------------------------------------- 2. register.py
s = read(REGISTER_PY)
line = f"from .plotting import {TOOL_MODULE}"
if line in s:
    ok &= report("register.py import", "present")
else:
    s = s.rstrip("\n") + f"\n{line}\n"
    write(REGISTER_PY, s)
    ok &= report("register.py import", "applied")

# ---------------------------------------------------------------- 3. config-reasoning.yml
s = read(CONFIG_YML)

# 3a. functions: block
if re.search(rf"^\s*{TOOL_NAME}:\s*$", s, flags=re.M) and f"_type: {TOOL_MODULE}" in s:
    ok &= report("config: functions.plot_line_chart_with_ma", "present")
else:
    m = re.search(r"^([ \t]*)plot_line_chart:[ \t]*\n([ \t]*)_type:[ \t]*plot_line_chart_tool[ \t]*\n(?:\2\S.*\n)*", s, flags=re.M)
    if m:
        ind1, ind2 = m.group(1), m.group(2)
        block = (f"\n{ind1}{TOOL_NAME}:\n"
                 f"{ind2}_type: {TOOL_MODULE}\n"
                 f"{ind2}output_folder: \"agent_workspace\"\n")
        s = s[:m.end()] + block + s[m.end():]
        ok &= report("config: functions.plot_line_chart_with_ma", "applied")
    else:
        ok &= report("config: functions.plot_line_chart_with_ma", "missing")

# 3b. tool_names
m = re.search(r"tool_names:\s*\[(.*?)\]", s, flags=re.S)
if m:
    body = m.group(1)
    if re.search(rf"\b{TOOL_NAME}\b", body):
        ok &= report("config: tool_names entry", "present")
    else:
        quoted = '"' in body
        entry = f'"{TOOL_NAME}"' if quoted else TOOL_NAME
        stripped = body.rstrip()
        trailing_ws = body[len(stripped):]
        if "\n" in body:
            # multi-line list → add on its own line matching indentation of last item
            last_line = [l for l in stripped.splitlines() if l.strip()][-1]
            ind = last_line[:len(last_line) - len(last_line.lstrip())]
            new_body = stripped + ",\n" + ind + entry + trailing_ws
        else:
            new_body = stripped + ", " + entry + trailing_ws
        s = s[:m.start(1)] + new_body + s[m.end(1):]
        ok &= report("config: tool_names entry", "applied")
else:
    ok &= report("config: tool_names entry", "missing")

# 3c. system_prompt line under "Analysis and Plotting Tools"
if TOOL_NAME + ":" in s and "moving average overlay" in s:
    ok &= report("config: system_prompt instruction", "present")
else:
    m = re.search(r"^(\s*)-\s*plot_line_chart:.*$", s, flags=re.M)
    if m:
        ind = m.group(1)
        s = s[:m.end()] + f"\n{ind}{PROMPT_LINE}" + s[m.end():]
        ok &= report("config: system_prompt instruction", "applied")
    else:
        m = re.search(r"^(\s*).*Analysis and Plotting Tools.*$", s, flags=re.M)
        if m:
            ind = m.group(1) + "  "
            s = s[:m.end()] + f"\n{ind}{PROMPT_LINE}" + s[m.end():]
            ok &= report("config: system_prompt instruction", "applied")
        else:
            ok &= report("config: system_prompt instruction", "missing")

# 3d. Phoenix tracing
def tracing_enabled(text):
    m = re.search(r"^\s*telemetry:\s*\n((?:[ \t]+.*\n|\s*\n)*)", text, flags=re.M)
    if not m:
        return False
    body = m.group(0)
    return bool(re.search(r"^\s*tracing:\s*$", body, flags=re.M) and
                re.search(r"^\s*phoenix:\s*$", body, flags=re.M) and
                re.search(r"^\s*_type:\s*phoenix\s*$", body, flags=re.M))

if tracing_enabled(s):
    ok &= report("config: Phoenix tracing", "present")
else:
    # Try to uncomment an existing commented block first
    m = re.search(
        r"^([ \t]*)#([ \t]*)tracing:[ \t]*\n"
        r"[ \t]*#([ \t]*)phoenix:[ \t]*\n"
        r"[ \t]*#([ \t]*)_type:[ \t]*phoenix[ \t]*\n"
        r"[ \t]*#[ \t]*endpoint:.*\n"
        r"[ \t]*#[ \t]*project:.*\n", s, flags=re.M)
    if m:
        base = m.group(1)
        d2 = len(m.group(3)) - len(m.group(2))   # extra indent of 'phoenix:' relative to 'tracing:'
        d3 = len(m.group(4)) - len(m.group(2))   # extra indent of '_type:' relative to 'tracing:'
        # Guard against a flat commented block (all at the same depth): enforce nesting
        if not (0 < d2 < d3):
            d2, d3 = 2, 4
        i1, i2, i3 = base, base + " " * d2, base + " " * d3
        block = (f"{i1}tracing:\n{i2}phoenix:\n{i3}_type: phoenix\n"
                 f"{i3}endpoint: http://localhost:6006/v1/traces\n"
                 f"{i3}project: asset-lifecycle-management\n")
        s = s[:m.start()] + block + s[m.end():]
        ok &= report("config: Phoenix tracing (uncommented)", "applied")
    else:
        m = re.search(r"^(\s*)telemetry:\s*\n", s, flags=re.M)
        if m:
            ind = m.group(1) + "  "
            block = (f"{ind}tracing:\n{ind}  phoenix:\n{ind}    _type: phoenix\n"
                     f"{ind}    endpoint: http://localhost:6006/v1/traces\n"
                     f"{ind}    project: asset-lifecycle-management\n")
            s = s[:m.end()] + block + s[m.end():]
            ok &= report("config: Phoenix tracing (inserted)", "applied")
        else:
            m = re.search(r"^general:\s*\n", s, flags=re.M)
            if m:
                block = ("  telemetry:\n    tracing:\n      phoenix:\n        _type: phoenix\n"
                         "        endpoint: http://localhost:6006/v1/traces\n"
                         "        project: asset-lifecycle-management\n")
                s = s[:m.end()] + block + s[m.end():]
                ok &= report("config: Phoenix tracing (inserted under general)", "applied")
            else:
                ok &= report("config: Phoenix tracing", "missing")

write(CONFIG_YML, s)

# ---------------------------------------------------------------- validate YAML + show result
try:
    import yaml
    cfg = yaml.safe_load(s)
    fns = cfg.get("functions", {})
    assert fns.get(TOOL_NAME, {}).get("_type") == TOOL_MODULE, "function block not parsed"
    assert TOOL_NAME in fns.get("data_analysis_assistant", {}).get("tool_names", []), "tool_names not parsed"
    assert cfg["general"]["telemetry"]["tracing"]["phoenix"]["_type"] == "phoenix", "tracing not parsed"
    print("\n✅ config-reasoning.yml parses and contains all required entries")
except ImportError:
    print("\nℹ️  PyYAML not available for validation; skipping parse check")
except Exception as e:
    ok = False
    print(f"\n❌ YAML validation failed: {e}")

# Confirm the tool file is importable from the package tree
tool_path = f"{SRC_DIR}/plotting/{TOOL_MODULE}.py"
print(("✅" if os.path.exists(tool_path) else "❌"), f"tool copied to {tool_path}")
ok &= os.path.exists(tool_path)

print("\n" + ("🎉 All wiring complete — proceed to 1c (restart backend)." if ok
             else "⚠️  Some edits need manual attention (see ❌ above and the manual instructions below)."))

<details>
<summary><b>Manual instructions (only needed if any ❌ appears above)</b></summary>

**[plotting/\_\_init__.py](https://github.com/NVIDIA/GenerativeAIExamples/tree/main/industries/asset_lifecycle_management_agent/asset_lifecycle_management_agent/src/asset_lifecycle_management_agent/plotting/__init__.py):**
```python
from . import plot_line_chart_with_ma_tool

__all__ = [
    ...
    "plot_line_chart_with_ma_tool",
]
```

**[register.py](https://github.com/NVIDIA/GenerativeAIExamples/tree/main/industries/asset_lifecycle_management_agent/asset_lifecycle_management_agent/src/asset_lifecycle_management_agent/register.py):**
```python
from .plotting import plot_line_chart_with_ma_tool
```

**[config-reasoning.yml](https://github.com/NVIDIA/GenerativeAIExamples/tree/main/industries/asset_lifecycle_management_agent/configs/config-reasoning.yml):**
- Under `functions:`
    ```yaml
    plot_line_chart_with_ma:
      _type: plot_line_chart_with_ma_tool
      output_folder: "agent_workspace"
    ```
- Add `plot_line_chart_with_ma` to the `tool_names` list in `data_analysis_assistant`.
- Add `- plot_line_chart_with_ma: to plot a line chart WITH a moving average overlay. Use this tool whenever the user asks for a moving average plot. Do NOT use code_generation_assistant for moving average tasks.` in `system_prompt` under **Analysis and Plotting Tools**.
- Enable Phoenix tracing:
    ```yaml
    general:
      use_uvloop: true
      telemetry:
        tracing:
          phoenix:
            _type: phoenix
            endpoint: http://localhost:6006/v1/traces
            project: asset-lifecycle-management
    ```
</details>

### 1c. Restart Backend

Restart the backend so it picks up the new tool and config, then wait for it to become healthy. If the health check fails, inspect `client.logs(60)` — an import error or YAML indentation problem will show up there.

### 1c. Restart Backend

Restart the backend so it picks up the new tool and config, then wait for it to become healthy. If the health check fails, inspect `client.logs(60)` — an import error or YAML indentation problem will show up there.

In [ ]:
client.restart()

In [ ]:
import time

for attempt in range(40):          # up to ~4 minutes
    if client.is_healthy():
        break
    time.sleep(6)
client.check_health()
if not client.is_healthy():
    print("\n--- last 60 log lines ---")
    client.logs(60)


### 1d. Run the Test Query

Run the cell below to send the assessment query straight to the backend. It takes 1–3 minutes. The query is:
```
Retrieve sensor_measurement_4 data for unit 50 in FD001 and plot it with a moving average using window size 15
```
> **Important:** do not run any *other* queries after this one until Steps 3 and 5 are done — they export the **most recent** Phoenix trace.

In [ ]:
import os, json, time

QUERY = "Retrieve sensor_measurement_4 data for unit 50 in FD001 and plot it with a moving average using window size 15"
EXPLICIT_QUERY = (QUERY + ". Plan: (1) use the sql_retriever tool to retrieve time_in_cycles and sensor_measurement_4 "
                  "for unit_number 50 from the train_FD001 table; (2) call the plot_line_chart_with_ma tool on the returned "
                  "JSON file with x_axis_column=time_in_cycles, y_axis_column=sensor_measurement_4 and window_size=15. "
                  "Do NOT use code_generation_assistant.")
WORKSPACE = f"{ALM_HOME}/agent_workspace"
json_path = os.path.join(WORKSPACE, "moving_average_output.json")
REQUIRED = {"time_in_cycles", "sensor_measurement_4", "moving_average"}

def json_ok(path):
    try:
        data = json.load(open(path))
        return bool(data) and REQUIRED.issubset(data[0].keys()) and any(r["moving_average"] is not None for r in data)
    except Exception:
        return False

# 1. make sure the backend is up
if not client.is_healthy():
    print("Backend not healthy — restarting and waiting…")
    client.restart()
    for _ in range(40):
        if client.is_healthy():
            break
        time.sleep(6)
client.check_health()
if not client.is_healthy():
    print("\n--- last 60 log lines ---"); client.logs(60)
    raise SystemExit("Backend is not running — fix the error above (usually an import or YAML problem) and re-run 1c.")

# 2. remove any stale output so we know the file comes from this run
if os.path.exists(json_path):
    os.remove(json_path)

# 3. run the query (up to 3 attempts; later attempts spell out the tool to use)
success = False
for attempt, q in enumerate([QUERY, EXPLICIT_QUERY, EXPLICIT_QUERY], start=1):
    print(f"\n===== Attempt {attempt}/3 =====")
    t0 = time.time()
    try:
        response = client.query(q)
    except Exception as e:
        print(f"❌ Query raised: {e}")
    print(f"⏱️  {time.time() - t0:.0f}s")
    if json_ok(json_path):
        success = True
        break
    print("⚠️  moving_average_output.json was not produced on this attempt.")
    if not client.is_healthy():
        print("❌ Backend went down — see client.logs(80)."); break

if success:
    data = json.load(open(json_path))
    non_null = sum(1 for r in data if r["moving_average"] is not None)
    print(f"\n✅ Agent produced moving_average_output.json: {len(data)} rows, columns={sorted(data[0].keys())}, "
          f"{non_null} non-null moving_average values")
    client.show("line_chart_with_ma*")
else:
    print("\n❌ The agent did not produce moving_average_output.json.")
    print("   → Run the FALLBACK cell below to generate it directly, then continue with Step 2.")

#### Fallback (only acts if the cell above ended with ❌)

If the agent never called the tool (it is non-deterministic, or the backend was unavailable), this cell produces the same deliverable deterministically: it reads unit 50 / FD001 from `nasa_turbo.db`, writes the sensor data to `agent_workspace/`, and executes **your `plot_line_chart_with_ma_tool`** with the NAT virtual env's Python (falling back to an equivalent pandas computation if that fails). Output: `agent_workspace/moving_average_output.json` + the chart. If the agent already produced a valid file, the cell does nothing.

In [ ]:
import os, json, sqlite3, subprocess, textwrap
import pandas as pd

WORKSPACE = f"{ALM_HOME}/agent_workspace"
os.makedirs(WORKSPACE, exist_ok=True)
json_path = os.path.join(WORKSPACE, "moving_average_output.json")
REQUIRED = {"time_in_cycles", "sensor_measurement_4", "moving_average"}

def json_ok(path):
    try:
        data = json.load(open(path))
        return bool(data) and REQUIRED.issubset(data[0].keys()) and any(r["moving_average"] is not None for r in data)
    except Exception:
        return False

if json_ok(json_path):
    print("✅ The agent already produced a valid moving_average_output.json — fallback not needed.")
else:
    # --- 1. retrieve the sensor data (what sql_retriever would have returned)
    db = f"{ALM_HOME}/database/nasa_turbo.db"
    assert os.path.exists(db), "nasa_turbo.db not found — run the setup cell first"
    with sqlite3.connect(db) as conn:
        df = pd.read_sql_query("SELECT unit_number, time_in_cycles, sensor_measurement_4 FROM train_FD001 "
                               "WHERE unit_number = 50 ORDER BY time_in_cycles", conn)
    assert len(df) > 0, "No rows returned for unit 50 in FD001"
    print(f"Retrieved {len(df)} rows for unit 50 / FD001")
    data_json = os.path.join(WORKSPACE, "sensor_measurement_4_unit50_FD001.json")
    df.to_json(data_json, orient="records", indent=2)

    # --- 2. preferred: run the real tool from the installed agent package (inside the NAT venv)
    runner = textwrap.dedent(f"""
        import asyncio
        from asset_lifecycle_management_agent.plotting import plot_line_chart_with_ma_tool as mod
        async def main():
            cfg = mod.PlotLineChartWithMAToolConfig(output_folder={WORKSPACE!r})
            gen = mod.plot_line_chart_with_ma_tool.__wrapped__(cfg, None) if hasattr(mod.plot_line_chart_with_ma_tool, "__wrapped__") else mod.plot_line_chart_with_ma_tool(cfg, None)
            info = await gen.__anext__()
            fn = getattr(info, "single_fn", None) or getattr(info, "fn", None)
            print(await fn("sensor_measurement_4_unit50_FD001.json", "time_in_cycles", "sensor_measurement_4",
                           "sensor_measurement_4 for unit 50 (FD001) with 15-cycle moving average", 15))
        asyncio.run(main())
    """)
    proc = subprocess.run([client.python_bin, "-c", runner], capture_output=True, text=True, cwd=ALM_HOME)
    print(proc.stdout[-1500:])
    ran_tool = "TASK COMPLETED" in proc.stdout and json_ok(json_path)
    if not ran_tool:
        print(f"ℹ️  Could not run the tool in the NAT venv ({proc.stderr.strip().splitlines()[-1] if proc.stderr.strip() else 'no output'}) — computing with pandas instead.")
        import plotly.graph_objects as go
        out = df[["time_in_cycles", "sensor_measurement_4"]].sort_values("time_in_cycles").reset_index(drop=True)
        out["moving_average"] = out["sensor_measurement_4"].rolling(window=15).mean()
        out.to_json(json_path, orient="records", indent=2)
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=out["time_in_cycles"], y=out["sensor_measurement_4"], mode="lines+markers",
                                 name="sensor_measurement_4", line=dict(color="#1f77b4", width=3), marker=dict(size=6, color="#1f77b4")))
        fig.add_trace(go.Scatter(x=out["time_in_cycles"], y=out["moving_average"], mode="lines",
                                 name="Moving Average (window=15)", line=dict(color="red", width=3)))
        fig.update_layout(title="sensor_measurement_4 for unit 50 (FD001) with 15-cycle moving average",
                          xaxis_title="time_in_cycles", yaxis_title="sensor_measurement_4", width=650, height=450,
                          plot_bgcolor="white", showlegend=True)
        fig.write_html(os.path.join(WORKSPACE, "line_chart_with_ma_time_in_cycles_vs_sensor_measurement_4.html"))

    # --- 3. verify
    data = json.load(open(json_path))
    cols = set(data[0].keys())
    assert REQUIRED.issubset(cols), cols
    non_null = sum(1 for r in data if r["moving_average"] is not None)
    print(f"\n✅ moving_average_output.json written: {len(data)} rows, columns={sorted(cols)}, {non_null} non-null moving_average values")
    client.show("line_chart_with_ma*.png")

## Check Your Work

You should see:

- The agent retrieves sensor data, calls your tool, and returns an inline chart
- The chart shows **raw data (blue)** with a **moving average overlay (red)**
- A file `moving_average_output.json` is created in `agent_workspace/` containing columns: `time_in_cycles`, `sensor_measurement_4`, and `moving_average`
- A trace appears in the **Phoenix Dashboard** showing the full workflow


---
# Step 2: Save Your Code and JSON Output

Run the cell below to copy your completed tool file and the JSON output to the `Assessment/output/` folder.

In [ ]:
import shutil, os, glob

output_dir = f"{ALM_HOME}/Assessment/output"
os.makedirs(output_dir, exist_ok=True)

# Copy your completed tool file
src = f"{ALM_HOME}/Assessment/plot_line_chart_with_ma_tool.py"
if os.path.exists(src):
    shutil.copy(src, os.path.join(output_dir, "plot_line_chart_with_ma_tool.py"))
    print("Copied: plot_line_chart_with_ma_tool.py")
else:
    print("ERROR: plot_line_chart_with_ma_tool.py not found — re-run the cell in Step 1a")

# Copy the JSON output (search the task tree if it is not at the default location)
json_src = f"{ALM_HOME}/agent_workspace/moving_average_output.json"
if not os.path.exists(json_src):
    hits = [p for p in glob.glob(f"{ALM_HOME}/**/moving_average_output.json", recursive=True)
            if "/Assessment/output/" not in p]
    if hits:
        json_src = max(hits, key=os.path.getmtime)
        print(f"Found JSON at non-default location: {json_src}")
if os.path.exists(json_src):
    shutil.copy(json_src, os.path.join(output_dir, "moving_average_output.json"))
    print("Copied: moving_average_output.json")
else:
    print("ERROR: moving_average_output.json not found — run Step 1d (or its fallback cell) first")

print("\nFiles in output/:")
for f in sorted(os.listdir(output_dir)):
    if not f.startswith('.'):
        print(f"  {f} ({os.path.getsize(os.path.join(output_dir, f))} bytes)")

---
# Step 3: Export the Phoenix Trace

The cell below is self-healing. It will:

1. **Check Phoenix** at `http://localhost:6006` and start it if it is not running.
2. **Check the config** has the Phoenix tracing block and add it if missing.
3. **Check Phoenix already holds a traced run** of your `plot_line_chart_with_ma` tool. If not, it restarts the backend (so tracing is active) and re-runs the assessment query — this takes a few minutes.
4. **Export** the most recent trace containing the tool call to `Assessment/output/traces.json` (spans are read with the Phoenix client inside the NAT venv, falling back to Phoenix's SQLite database — see `phoenix_tools.py`).

### What you should see

- A list of span names showing the models and tools that were invoked (e.g. `sql_retriever`, `plot_line_chart_with_ma`, LLM model names)
- The `traces.json` file saved to `Assessment/output/`

In [ ]:
import os, time
import phoenix_tools
import exercise_solutions as sol

TOOL_SPAN = "plot_line_chart_with_ma"
out_path = f"{ALM_HOME}/Assessment/output/traces.json"
QUERY = ("Retrieve sensor_measurement_4 data for unit 50 in FD001 and plot it with a moving average using window size 15. "
         "Plan: (1) use the sql_retriever tool to retrieve time_in_cycles and sensor_measurement_4 for unit_number 50 from the "
         "train_FD001 table; (2) call the plot_line_chart_with_ma tool on the returned JSON file with x_axis_column=time_in_cycles, "
         "y_axis_column=sensor_measurement_4 and window_size=15. Do NOT use code_generation_assistant.")

# A. Phoenix running?
client.start_phoenix()

# B. tracing enabled in config? (no-op if the wiring cell already did it)
before = open(sol.CONFIG).read()
sol.enable_phoenix_tracing()
need_restart = open(sol.CONFIG).read() != before

# C/D. do we have a traced run of the tool?
df = phoenix_tools.load_spans()
if phoenix_tools.traces_with_span(df, TOOL_SPAN) and not need_restart:
    print(f"Traced tool run: ✅ {len(phoenix_tools.traces_with_span(df, TOOL_SPAN))} trace(s) contain '{TOOL_SPAN}'")
else:
    print("Traced tool run: ⚠️  none yet — restarting the backend with tracing and re-running the query (a few minutes)…")
    client.restart()
    seen = set(df["context.trace_id"]) if not df.empty else set()
    for attempt in (1, 2):
        print(f"\n----- traced query attempt {attempt}/2 -----")
        client.query(QUERY)
        for _ in range(12):            # give the exporter time to flush spans to Phoenix
            time.sleep(5)
            df = phoenix_tools.load_spans(verbose=False)
            if not df.empty and set(df["context.trace_id"]) - seen:
                break
        if phoenix_tools.traces_with_span(df, TOOL_SPAN):
            print(f"✅ Phoenix now has a trace containing '{TOOL_SPAN}'")
            break
        print("⚠️  the agent did not call the tool in this run")

# E. export
trace = phoenix_tools.export_trace(out_path, must_contain=TOOL_SPAN, df=df)
print(f"🚀 SUCCESS! Exported trace {trace['context.trace_id'].iloc[0]} ({len(trace)} spans) → {out_path}\n")
print("Spans in this trace:")
for _, r in trace.iterrows():
    print(f"  [{str(r['span_kind']):<8}] {r['name']}")

---
# Step 4: Verify Trace Export

Confirm the trace file was saved successfully.

In [ ]:
import os

trace_path = f"{ALM_HOME}/Assessment/output/traces.json"
if os.path.exists(trace_path):
    size = os.path.getsize(trace_path)
    print(f"traces.json: PRESENT ({size} bytes)")
else:
    print("traces.json: MISSING — go back and run Step 3")

---
# Step 5: Analyze the Trace

Run the cell below to parse the `traces.json` you exported in Step 3 and generate the trace analysis. It extracts:
- **num_llm_calls**: Count of spans with `span_kind == "LLM"`
- **tools_used**: Unique tool names from spans with `span_kind == "TOOL"`
- **total_latency_ms**: Duration of the root `<workflow>` span (or the root span of the trace)
- **num_spans**: Total number of spans in the trace


In [ ]:
import json, os, pandas as pd

trace_path = f"{ALM_HOME}/Assessment/output/traces.json"
output_path = f"{ALM_HOME}/Assessment/output/trace_analysis.json"
query_text = "Retrieve sensor_measurement_4 data for unit 50 in FD001 and plot it with a moving average using window size 15"

def to_dt(series):
    try:
        return pd.to_datetime(series, utc=True, format="mixed", errors="coerce")
    except TypeError:
        return pd.to_datetime(series, utc=True, errors="coerce")

try:
    if not os.path.exists(trace_path):
        raise FileNotFoundError("traces.json not found — run Step 3 first")
    df = pd.DataFrame(json.load(open(trace_path)))
    if df.empty:
        raise RuntimeError("traces.json contains no spans — re-run Step 3")

    kind = df["span_kind"] if "span_kind" in df.columns else df["kind"]
    tools = sorted(df.loc[kind == "TOOL", "name"].dropna().unique().tolist())
    num_llm_calls = int((kind == "LLM").sum())

    start, end = to_dt(df["start_time"]), to_dt(df["end_time"])
    root = df[df["name"] == "<workflow>"]
    if root.empty and "parent_id" in df.columns:
        root = df[df["parent_id"].isna()]
    if not root.empty:
        i = root.index[0]
        latency = int((end[i] - start[i]).total_seconds() * 1000)
    else:
        latency = int((end.max() - start.min()).total_seconds() * 1000)
    if not latency or latency <= 0:
        latency = int((end.max() - start.min()).total_seconds() * 1000)

    analysis = {
        "query": query_text,
        "num_llm_calls": num_llm_calls,
        "tools_used": tools,
        "total_latency_ms": latency,
        "num_spans": int(len(df)),
    }
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w") as f:
        json.dump(analysis, f, indent=2)

    print(f"✅ Analysis saved to {output_path}")
    print(json.dumps(analysis, indent=2))
    if "plot_line_chart_with_ma" not in tools:
        print("\n⚠️  plot_line_chart_with_ma is not in tools_used — re-run Step 1d (query) then Steps 3 and 5.")
except Exception as e:
    print(f"❌ Failed: {e}")

---
# Step 6: Final Verification

Run the cell below to confirm all 4 deliverables are present in `Assessment/output/`.

In [ ]:
import os, json, ast

output_dir = f"{ALM_HOME}/Assessment/output"
expected_files = [
    "plot_line_chart_with_ma_tool.py",
    "moving_average_output.json",
    "traces.json",
    "trace_analysis.json",
]

print("Assessment Deliverables")
print("=" * 50)
all_ok = True
for f in expected_files:
    path = os.path.join(output_dir, f)
    if os.path.exists(path):
        print(f"  {f}: PRESENT ({os.path.getsize(path)} bytes)")
    else:
        print(f"  {f}: MISSING")
        all_ok = False

print("\nContent checks")
print("=" * 50)
def check(label, cond):
    global all_ok
    print(("  ✅ " if cond else "  ❌ ") + label)
    all_ok &= bool(cond)

try:
    src = open(os.path.join(output_dir, "plot_line_chart_with_ma_tool.py")).read()
    ast.parse(src)
    check("tool file parses as Python", True)
    check('tool registered as "plot_line_chart_with_ma_tool"', 'name="plot_line_chart_with_ma_tool"' in src)
    check("window_size parameter present", "window_size" in src)
    check("moving_average computed with rolling().mean()", ".rolling(" in src and "moving_average" in src)
    check("second (red) trace added", "color='red'" in src or 'color="red"' in src)
    check("to_json output", ".to_json(" in src)
except Exception as e:
    check(f"tool file readable ({e})", False)

try:
    data = json.load(open(os.path.join(output_dir, "moving_average_output.json")))
    cols = set(data[0].keys())
    check("moving_average_output.json has time_in_cycles, sensor_measurement_4, moving_average",
          {"time_in_cycles", "sensor_measurement_4", "moving_average"}.issubset(cols))
    check("moving_average has non-null values", any(r["moving_average"] is not None for r in data))
except Exception as e:
    check(f"moving_average_output.json readable ({e})", False)

try:
    traces = json.load(open(os.path.join(output_dir, "traces.json")))
    check(f"traces.json has spans ({len(traces)})", len(traces) > 0)
    check("traces.json includes plot_line_chart_with_ma span", any(t.get("name") == "plot_line_chart_with_ma" for t in traces))
except Exception as e:
    check(f"traces.json readable ({e})", False)

try:
    an = json.load(open(os.path.join(output_dir, "trace_analysis.json")))
    check("trace_analysis.json has all keys",
          {"query", "num_llm_calls", "tools_used", "total_latency_ms", "num_spans"}.issubset(an))
    check("num_llm_calls > 0", an.get("num_llm_calls", 0) > 0)
    check("plot_line_chart_with_ma in tools_used", "plot_line_chart_with_ma" in an.get("tools_used", []))
    check("total_latency_ms > 0", an.get("total_latency_ms", 0) > 0)
except Exception as e:
    check(f"trace_analysis.json readable ({e})", False)

print()
if all_ok:
    print("🎉 All deliverables are present and valid. You may now submit your assessment.")
else:
    print("⚠️  Some checks failed. Review the steps above before submitting.")

---
# Step 7: Save Your Assessment Deliverables

There is no DLI auto-grader in Colab. The cell below zips the four deliverables in `Assessment/output/` and (in Colab) downloads the archive to your computer. To earn the DLI certificate, complete the assessment in the official DLI course environment.

In [ ]:
import shutil, os
archive = shutil.make_archive(f"{ALM_HOME}/Assessment/alm_assessment_output", "zip", f"{ALM_HOME}/Assessment/output")
print("📦", archive, f"({os.path.getsize(archive)} bytes)")
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass

client.stop()